In [ ]:
# forbidden access (i think just set the flag upon super.init)
# because the flag prevents access of the super methods (and i don't think it's as much of a problem in this implementation)
# build dm

In [ ]:
"""
sandbox_time.ipynb

A sandbox to develop a time-resolved class.

Author: Stellina X. Ao
Created: 2026-07-07
Last Modified: 2026-07-07
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

In [ ]:
"""--------------------------------------------"""
# SANITY CHECKS
# beta weight sanity checks
# -> multiply beta weight by 1/binwidth_s and make sure identical to firing rate
"""--------------------------------------------"""

## init

In [ ]:
from sg.models import make_tre, Encoder

encoder = make_tre(Encoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=False,
    stepsize_s=0.1,
)

"""
encoder_mb = make_tre(StrategyEncoder)(
    subj_id,
    sess_id,
    norm=False,
    stepsize_s=0.1,
    strategy_filter="mb",
)

encoder_mf = make_tre(StrategyEncoder)(
    subj_id,
    sess_id,
    norm=False,
    stepsize_s=0.1,
    strategy_filter="mf",
)
"""

In [ ]:
import numpy as np

X = np.reshape(np.arange(24), (2, 3, 4))
X, X.transpose(1, 0, 2).reshape(6, 4)

In [ ]:
encoder.robs_predict["baseline"]

In [ ]:
A = np.arange(10).reshape(2, 5)
print(A.shape)
B = np.repeat(A, 3, axis=0)
A, B
B.shape

In [ ]:
encoder.verify()

In [ ]:
plt.figure()
plt.imshow(encoder.tvs, aspect="auto", interpolation="none")
plt.show()

In [ ]:
import numpy as np

tvs = encoder.trial_data[encoder.tv_keys]
for k in encoder.tv_keys:
    if "rewarded" in k:
        tvs[k] = tvs[k].replace(0, -1)
tvs = np.array(tvs, dtype="float32")

In [ ]:
num_trials = 203

masks = np.zeros((encoder.num_bins, num_trials * encoder.num_bins))
for i in range(encoder.num_bins):
    motif = np.zeros(encoder.num_bins)
    motif[i] = 1
    masks[i] = np.tile(motif, num_trials)

In [ ]:
response = tvs[:, 0]
response_tr_interstitial = np.tile(
    np.repeat(response, encoder.num_bins), reps=(encoder.num_bins, 1)
)

response_tr = response_tr_interstitial * masks

plt.figure()
plt.imshow(response_tr, aspect="auto", interpolation="none")
plt.show()

In [ ]:
tv_tr_interstitial = np.tile(
    np.repeat(tvs, encoder.num_bins, axis=0), reps=(encoder.num_bins, 1, 1)
)
tv_tr = tv_tr_interstitial * masks[:, :, None]

tv_tr = tv_tr.transpose(2, 0, 1).reshape(-1, encoder.num_bins * num_trials).T

plt.figure()
plt.imshow(tv_tr.T, aspect="auto", interpolation="none")
plt.show()

In [ ]:
A = np.repeat(tvs, encoder.num_bins, axis=0)
B = np.tile(A, reps=(encoder.num_bins, 1, 1))
assert np.all(B[:, :, 0] == response_tr_interstitial)

C = B * masks[:, :, None]
assert np.all(C[:, :, 0] == response_tr)

In [ ]:
X = np.arange(24).reshape((2, 3, 4))

In [ ]:
203 * 15

In [ ]:
Y = C.transpose(2, 0, 1)
Z = Y.reshape(-1, Y.shape[-1]).T
Z

In [ ]:
C.shape

In [ ]:
# response_1, response_2, .., response_15, .., rewarded_prev_15

In [ ]:
X, Y

In [ ]:
encoder.get_data()

In [ ]:
encoder.plot_r2_distro()

In [ ]:
(
    encoder.psths["DLS"].shape,
    encoder.psths["DMS"].shape,
)

In [ ]:
# encoder.verify()
# encoder_mb.verify()
# encoder_mf.verify()

In [ ]:
encoder.tvs.shape

In [ ]:
import numpy as np

for key in encoder.tv_keys:
    print(key, np.unique(encoder.trial_data[key]))

## the t-population

In [ ]:
# select the neurons that lie along the axis
# plot their encoding for mb/mf
# is it just a different encoding pattern (i.e., they encode different things)
# or are they silent(er) in another strategy